# NASA EPSCoR Progress: PAX1000IR Python Interfacing
### 20260723 - *Tyndale Stutzman*
---

## Introduction:

This notebook is intended to provide a testbed for interfacing the **Thorlabs PAX1000IR1** polarimeter. Unfortunately, the proprietary GUI software for this device is only available through Windows, therefore, exposing the raw data through python is necessary in order to use this polarimeter with Pyrpl and other python experiments. 

In [1]:
import usb.core

dev = usb.core.find(idVendor=0x1313, idProduct=0x8031)
print(dev)

DEVICE ID 1313:8031 on Bus 003 Address 006 =================
 bLength                :   0x12 (18 bytes)
 bDescriptorType        :    0x1 Device
 bcdUSB                 :  0x200 USB 2.0
 bDeviceClass           :    0x0 Specified at interface
 bDeviceSubClass        :    0x0
 bDeviceProtocol        :    0x0
 bMaxPacketSize0        :   0x40 (64 bytes)
 idVendor               : 0x1313
 idProduct              : 0x8031
 bcdDevice              :  0x121 Device 1.21
 iManufacturer          :    0x1 Error Accessing String
 iProduct               :    0x2 Error Accessing String
 iSerialNumber          :    0x3 Error Accessing String
 bNumConfigurations     :    0x1
  CONFIGURATION 1: 500 mA ==================================
   bLength              :    0x9 (9 bytes)
   bDescriptorType      :    0x2 Configuration
   wTotalLength         :   0x30 (48 bytes)
   bNumInterfaces       :    0x2
   bConfigurationValue  :    0x1
   iConfiguration       :    0x0 
   bmAttributes         :   0x80 Bus Powe

In [2]:
for cfg in dev:
    print("Configuration", cfg.bConfigurationValue)
    for intf in cfg:
        print(" Interface", intf.bInterfaceNumber)
        for ep in intf:
            print(
                "  Endpoint",
                hex(ep.bEndpointAddress),
                "Attributes",
                hex(ep.bmAttributes)
            )

Configuration 1
 Interface 0
  Endpoint 0x5 Attributes 0x2
  Endpoint 0x82 Attributes 0x2
  Endpoint 0x87 Attributes 0x3
 Interface 1


In [4]:
import pyvisa

rm = pyvisa.ResourceManager('@py')

print(rm.list_resources())

()


****Fix Device Permissions****

In order to communicate with the PAX1000 it might be necessary to change the device permissions. The following lines were attempted:

Create new udev rule:
```sudo nano /etc/udev/rules.d/99-thorlabs-pax.rules```

Paste:
```SUBSYSTEM=="usbmisc", ATTRS{idVendor}=="1313", ATTRS{idProduct}=="8031", MODE="0666"```

Then reload:
```sudo udevadm control --reload-rules sudo udevadm trigger```

Finally, verify the new rules with ```ls -l /dev/usbtmc0``` which should yeild something like ```crw-rw-rw-```

#### NI-VISA yaq Daemon

In [12]:
import pyvisa

rm = pyvisa.ResourceManager()

print(rm.visalib.library_path)
print(rm.list_resources())

/usr/lib/x86_64-linux-gnu/libvisa.so.0.0.0
('USB3::0x1313::0x8031::M01303584',)


In [1]:
import pyvisa

rm = pyvisa.ResourceManager()

session, status = rm.open_bare_resource(
    "USB3::0x1313::0x8031::M01303584"
)

print(session, status)

# test write/read directly
rm.visalib.write(session, b"*IDN?\n")

data, status = rm.visalib.read(session, 4096)

print(data.decode(errors="replace"))

2 StatusCode.success
THORLABS,PAX1000IR1,M01303584,1.2.1



In [1]:
import pyvisa

resource = "USB3::0x1313::0x8031::M01303584"

rm = pyvisa.ResourceManager()

session, status = rm.open_bare_resource(resource)

print(session, status)

# write *IDN?
rm.visalib.write(session, b"*IDN?\n")

# read response
response, status = rm.visalib.read(session, 1024)

print(response.decode())

rm.visalib.close(session)
rm.close()

2 StatusCode.success
THORLABS,PAX1000IR1,M01303584,1.2.1



#### Current Stack:

Run the **yaq** client in the `.yoml` directory with `yaqd-thorlabs-pax1000 -c pax1000.toml -v

In [ ]:
# !/bin/bash

python - <<'EOF'
import yaqc

c = yaqc.Client(host="localhost", port=38400)

print([x for x in dir(c) if not x.startswith("_")])
EOF

"""
['busy', 'clear_connection_callbacks', 'direct_serial_write', 'get_channel_names', 'get_channel_shapes', 'get_channel_units', 'get_config', 'get_config_filepath', 'get_measured', 'get_measurement_id', 'get_state', 'get_wavelength', 'handshake', 'id', 'measure', 'properties', 'register_connection_callback', 'send', 'set_wavelength', 'shutdown', 'stop_looping', 'traits']
"""


In [ ]:
# !/bin/bash

python - <<'EOF'
import yaqc

c = yaqc.Client(host="localhost", port=38400)

print(c.traits)
print(c.properties)
EOF
['has-measure-trigger', 'is-daemon', 'is-sensor', 'uses-serial']
<yaqc._dotdict.DotDict object at 0x7db933ef7bb0>


In [ ]:
# !/bin/bash

python - <<'EOF'
import yaqc

c = yaqc.Client(host="localhost", port=38400)

print("channels:")
print(c.get_channel_names())

print("shapes:")
print(c.get_channel_shapes())

print("units:")
print(c.get_channel_units())
EOF
channels:
['revisions', 'timestamp', 'adc_min', 'adc_max', 'rev_time', 'theta', 'eta', 'dop', 'ptotal']
shapes:
{'revisions': [], 'timestamp': [], 'adc_min': [], 'adc_max': [], 'rev_time': [], 'theta': [], 'eta': [], 'dop': [], 'ptotal': []}
units:
{'revisions': None, 'timestamp': None, 'adc_min': None, 'adc_max': None, 'rev_time': None, 'theta': 'radian', 'eta': 'radian', 'dop': None, 'ptotal': None}


#### PAX1000 Measurement Channels

| Parameter | Description |
|-----------|-------------|
| `theta`   | polarization azimuth angle |
| `eta`     | ellipticity angle |
| `dop`     | degree of polarization |
| `ptotal`  | total optical power |

...following the architecture:

```mermaid
flowchart TD
    A["Notebook / script"]
    B["yaqc"]
    C["yaqd-thorlabs-pax1000"]
    D["NI-VISA"]
    E["PAX1000"]

    A --> B
    B --> C
    C --> D
    D --> E

    N["Owns VISA session<br/>for entire lifetime"] -.-> C
```

In [2]:
# FIRST IN-NOTEBOOK MEASUREMENT

import yaqc
import time

# Connect to running yaqd daemon
pax = yaqc.Client(host="localhost", port=38400)

# Trigger a measurement
measurement_id = pax.measure()

# Give the daemon time to acquire data
time.sleep(1)

# Retrieve result
data = pax.get_measured()

print("Measurement ID:", measurement_id)
print()
print("PAX1000 Measurement:")
for key, value in data.items():
    print(f"{key:12s}: {value}")

Measurement ID: 3

PAX1000 Measurement:
revisions   : 76724.0
timestamp   : 2494700.0
adc_min     : 21488.0
adc_max     : 31552.0
rev_time    : 0.06063769
theta       : -1.521835
eta         : -0.369228
dop         : 0.170406
ptotal      : 0.0004802537
measurement_id: 3


In [3]:
# RUN CONTINUOUS

import yaqc
import time
import threading

# Connect to PAX1000 daemon
c = yaqc.Client(host="localhost", port=38400)

# Stop flag
monitor_running = True

def monitor_pax(interval=1.0):
    global monitor_running
    
    print("Starting PAX1000 live monitor. Set monitor_running = False to stop.")
    
    while monitor_running:
        try:
            c.measure()
            time.sleep(0.5)  # allow measurement to complete
            
            data = c.get_measured()
            
            print(
                f"theta={data['theta']:.5f} rad | "
                f"eta={data['eta']:.5f} rad | "
                f"DOP={data['dop']:.5f} | "
                f"P={data['ptotal']:.6e}"
            )
            
            time.sleep(interval)
            
        except Exception as e:
            print("Monitor error:", e)
            break

# Start background thread
monitor_thread = threading.Thread(
    target=monitor_pax,
    daemon=True
)

monitor_thread.start()

Starting PAX1000 live monitor. Set monitor_running = False to stop.
theta=-1.23931 rad | eta=-0.56004 rad | DOP=0.40646 | P=5.054641e-04
theta=-1.12113 rad | eta=-0.51779 rad | DOP=0.41172 | P=5.005884e-04
theta=-1.16882 rad | eta=-0.53263 rad | DOP=0.41933 | P=5.065314e-04
theta=-1.12556 rad | eta=-0.52571 rad | DOP=0.41409 | P=5.003676e-04
theta=-1.27417 rad | eta=-0.57618 rad | DOP=0.41709 | P=5.053288e-04
theta=-1.19273 rad | eta=-0.55954 rad | DOP=0.41308 | P=5.009458e-04
theta=-1.18603 rad | eta=-0.56848 rad | DOP=0.40869 | P=4.975010e-04
theta=-1.21397 rad | eta=-0.57374 rad | DOP=0.41331 | P=5.004932e-04
theta=-1.17912 rad | eta=-0.54154 rad | DOP=0.44362 | P=5.130037e-04
theta=-1.14659 rad | eta=-0.52597 rad | DOP=0.44084 | P=5.103158e-04
theta=-1.13021 rad | eta=-0.52395 rad | DOP=0.44345 | P=5.092125e-04


In [4]:
# STOP MONITORING

monitor_running = False
print("Stopping monitor...")

Stopping monitor...


### Real-Time Plotting (hopefully)

In [3]:
import yaqc
import time
import threading
import matplotlib.pyplot as plt
from collections import deque

# Connect
c = yaqc.Client(host="localhost", port=38400)

# storage
max_points = 200

times = deque(maxlen=max_points)
theta = deque(maxlen=max_points)
eta = deque(maxlen=max_points)
dop = deque(maxlen=max_points)
power = deque(maxlen=max_points)

start_time = time.time()

monitor_running = True


def acquisition_loop():
    global monitor_running
    
    while monitor_running:
        try:
            # trigger measurement
            c.measure()
            
            # wait just enough for measurement
            time.sleep(0.07)
            
            data = c.get_measured()
            
            t = time.time() - start_time
            
            times.append(t)
            theta.append(data["theta"])
            eta.append(data["eta"])
            dop.append(data["dop"])
            power.append(data["ptotal"])
            
        except Exception as e:
            print("Acquisition error:", e)
            break


thread = threading.Thread(
    target=acquisition_loop,
    daemon=True
)

thread.start()

print("Acquisition running")

Acquisition running
